# RQ2 — Graph Mining + Community Detection on GRBench

**Research question:** *What structural properties of keyword co-occurrence graphs differentiate domains, and do graph centrality measures correlate with question difficulty?*

**Pipeline:**
1. Build a **keyword co-occurrence graph per domain** (question-level document window, stopword-filtered unigrams, TF-based keyword selection).
2. Compute per-domain structural statistics (nodes, edges, density, clustering, components).
3. Build a **global keyword graph** over all 1,740 questions; compute top-15 PageRank and top-15 betweenness.
4. **Louvain community detection** per domain → modularity and community sizes.
5. **Centrality ↔ difficulty** correlation: per question, compute mean PageRank / betweenness of its keywords on the global graph; Spearman r with difficulty (easy=0, medium=1, hard=2).
6. **Word-frequency baseline**: TF-IDF unigrams → 5-fold Logistic Regression on difficulty; compare macro-F1 to RQ1's embeddings baseline.

**Collaboration declaration**
1. **Collaborators**: solo project.
2. **Web sources**: NetworkX docs, python-louvain README.
3. **AI tools**: Claude Code (Anthropic) wrote analysis code under the author's direction; all methodological choices reviewed.
4. **Citations**: Hagberg et al., *NetworkX*, SciPy 2008 · Page et al., *The PageRank Citation Ranking*, 1999 · Freeman, *A set of measures of centrality based on betweenness*, Sociometry 1977 · Blondel et al., *Fast unfolding of communities in large networks*, J. Stat. Mech. 2008 (Louvain) · Spearman, 1904.

In [1]:
from __future__ import annotations

import json
import re
import time
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import community as community_louvain
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="talk")

HANDOFF = Path("/home/ybai/CSCE676/handoff/rq2_graph_mining")
FIGS = HANDOFF / "figures"
TABLES = HANDOFF / "tables"
DATA_DIR = Path("/home/ybai/CSCE676/data/raw/grbench")
for d in (FIGS, TABLES):
    d.mkdir(parents=True, exist_ok=True)

wall_clock: dict[str, float] = {}

## 1. Load GRBench and tokenise

**Why question-level window?** GRBench questions are short (median ~20 words, max ~250 words; see CP1 EDA). Most questions are single sentences. A sentence-level window would collapse onto a document-level window for the vast majority of questions and add parsing complexity without new signal. We therefore treat each question as a single "document" and build co-occurrence edges between distinct keywords in the same question.

**Tokenisation**: lower-case, keep alphabetic tokens of length ≥ 3, strip scikit-learn's ENGLISH_STOP_WORDS (n=318), plus a small extension for question-template words ("which", "what", "show", "find", "list", "name", "give", "tell", "mention", "please", "specify"). Numbers and punctuation are dropped.

In [2]:
t0 = time.time()
rows: list[dict] = []
for p in sorted(DATA_DIR.glob("*.json")):
    for line in p.read_text().splitlines():
        if line.strip():
            r = json.loads(line)
            r["domain"] = p.stem
            rows.append(r)
df = pd.DataFrame(rows).reset_index(drop=True)
print("loaded:", df.shape)

extra_stop = {
    "which", "what", "show", "find", "list", "name", "give", "tell", "mention",
    "please", "specify", "provide", "does", "could", "would", "tell", "know",
    "want", "like", "need", "something", "somebody", "anything", "anyone",
}
STOPWORDS = set(ENGLISH_STOP_WORDS) | extra_stop
TOK_RE = re.compile(r"[a-zA-Z]{3,}")


def tokenize(text: str) -> list[str]:
    toks = [t.lower() for t in TOK_RE.findall(text)]
    return [t for t in toks if t not in STOPWORDS]


df["tokens"] = df["question"].apply(tokenize)
print("avg tokens/question:", df["tokens"].apply(len).mean().round(2))
wall_clock["tokenize"] = time.time() - t0

loaded: (1740, 5)
avg tokens/question: 15.07


## 2. Per-domain keyword co-occurrence graphs

For each domain we:
1. Keep keywords that appear in **≥ 3 questions** (domain-level document frequency ≥ 3) — removes hapax legomena that otherwise inflate node count with noise.
2. Connect two keywords with an edge when they appear in the same question; edge weight = number of questions they co-occur in.
3. Compute: node count, edge count, density, average clustering coefficient, connected-component count, largest-component fraction, mean degree.

**Why ≥ 3 DF?** CP2 used no cutoff and produced noisy graphs dominated by single-question trivia. ≥ 3 is the smallest cutoff that still keeps recurring domain vocabulary while removing once-off entities. Sensitivity: increasing to ≥ 5 shrinks nodes by ~40% but preserves structural ranking across domains (checked in development).

In [3]:
t0 = time.time()

def build_cooccurrence(token_lists: list[list[str]], min_df: int = 3) -> nx.Graph:
    doc_freq: Counter[str] = Counter()
    for toks in token_lists:
        for t in set(toks):
            doc_freq[t] += 1
    keep = {t for t, c in doc_freq.items() if c >= min_df}
    g = nx.Graph()
    g.add_nodes_from(keep)
    edge_w: defaultdict[tuple[str, str], int] = defaultdict(int)
    for toks in token_lists:
        uniq = sorted({t for t in toks if t in keep})
        for a, b in combinations(uniq, 2):
            edge_w[(a, b)] += 1
    for (a, b), w in edge_w.items():
        g.add_edge(a, b, weight=w)
    return g


per_domain_stats = []
domain_graphs: dict[str, nx.Graph] = {}
for domain, grp in df.groupby("domain"):
    g = build_cooccurrence(grp["tokens"].tolist(), min_df=3)
    domain_graphs[domain] = g
    cc = list(nx.connected_components(g))
    largest = max((len(c) for c in cc), default=0)
    per_domain_stats.append(
        {
            "domain": domain,
            "n_questions": len(grp),
            "nodes": g.number_of_nodes(),
            "edges": g.number_of_edges(),
            "density": nx.density(g) if g.number_of_nodes() > 1 else 0.0,
            "avg_clustering": nx.average_clustering(g) if g.number_of_nodes() else 0.0,
            "n_components": len(cc),
            "largest_component_frac": largest / g.number_of_nodes() if g.number_of_nodes() else 0.0,
            "avg_degree": float(np.mean([d for _, d in g.degree()])) if g.number_of_nodes() else 0.0,
        }
    )
stats_df = pd.DataFrame(per_domain_stats).sort_values("nodes", ascending=False).reset_index(drop=True)
stats_df.to_csv(TABLES / "per_domain_graph_stats.csv", index=False)
print(stats_df.round(3).to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col, title in zip(
    axes.flat,
    ["nodes", "density", "avg_clustering", "largest_component_frac"],
    ["Node count", "Density", "Avg clustering coefficient", "Largest component fraction"],
):
    order = stats_df.sort_values(col, ascending=False)["domain"]
    sns.barplot(data=stats_df, x="domain", y=col, order=order, color="#4c72b0", ax=ax)
    ax.set(title=title, xlabel="", ylabel="")
    ax.tick_params(axis="x", rotation=45)
    for label in ax.get_xticklabels():
        label.set_ha("right")
plt.tight_layout()
fig.savefig(FIGS / "graph_stats_by_domain.png", dpi=180, bbox_inches="tight")
plt.close(fig)
wall_clock["per_domain_graphs"] = time.time() - t0

           domain  n_questions  nodes  edges  density  avg_clustering  n_components  largest_component_frac  avg_degree
            legal          180    364  15668    0.237           0.589             1                   1.000      86.088
materials_science          140    167   4746    0.342           0.694             1                   1.000      56.838
        chemistry          140    157   3951    0.323           0.705             1                   1.000      50.331
         medicine          140    154   4133    0.351           0.719             1                   1.000      53.675
          biology          140    146   3520    0.333           0.723             1                   1.000      48.219
          physics          140    131   2448    0.287           0.693             1                   1.000      37.374
       healthcare          270    127    693    0.087           0.653             1                   1.000      10.913
 computer_science          150    124   

## 3. Global keyword graph — top-15 PageRank and betweenness

The global graph pools all 1,740 questions. Same min_df = 3 rule applied globally. PageRank uses edge weights (co-occurrence counts); betweenness is unweighted because weighted betweenness on this graph inverts the semantic of weight (higher weight = closer, but shortest-path costs lower distances). Both are normalised by NetworkX.

In [4]:
t0 = time.time()
global_g = build_cooccurrence(df["tokens"].tolist(), min_df=3)
print("global graph:", global_g.number_of_nodes(), "nodes,", global_g.number_of_edges(), "edges")

pr = nx.pagerank(global_g, weight="weight", max_iter=500)
bc = nx.betweenness_centrality(global_g, k=min(500, global_g.number_of_nodes()), seed=SEED)

pr_top = pd.Series(pr).sort_values(ascending=False).head(15).rename("pagerank")
bc_top = pd.Series(bc).sort_values(ascending=False).head(15).rename("betweenness")

centrality_df = pd.DataFrame({
    "rank": range(1, 16),
    "pagerank_keyword": pr_top.index,
    "pagerank_value": pr_top.values,
    "betweenness_keyword": bc_top.index,
    "betweenness_value": bc_top.values,
})
centrality_df.to_csv(TABLES / "global_centrality_top15.csv", index=False)
print(centrality_df.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(data=pr_top.reset_index(), x="pagerank", y="index", color="#4c72b0", ax=axes[0])
axes[0].set(title="Top-15 PageRank (global, weighted)", xlabel="PageRank", ylabel="")
sns.barplot(data=bc_top.reset_index(), x="betweenness", y="index", color="#dd8452", ax=axes[1])
axes[1].set(title="Top-15 Betweenness (global, unweighted)", xlabel="Betweenness centrality", ylabel="")
plt.tight_layout()
fig.savefig(FIGS / "global_centrality.png", dpi=180, bbox_inches="tight")
plt.close(fig)
wall_clock["global_centrality"] = time.time() - t0

global graph: 1675 nodes, 104674 edges


 rank pagerank_keyword  pagerank_value betweenness_keyword  betweenness_value
    1            paper          0.0136               paper             0.0641
    2            title          0.0073              papers             0.0336
    3          opinion          0.0068              number             0.0268
    4            court          0.0063              author             0.0236
    5           papers          0.0061                case             0.0186
    6           author          0.0060               based             0.0177
    7           number          0.0058                 new             0.0176
    8           reader          0.0049               title             0.0173
    9             case          0.0044           following             0.0143
   10          cluster          0.0039         recommended             0.0125
   11            study          0.0039               court             0.0121
   12           effect          0.0037             disease      

## 4. Louvain community detection per domain

Louvain maximises modularity over a partition of the graph. Reported: number of communities, modularity Q, and the fraction of nodes in the largest community (a concentration proxy). Random seed passed so the non-deterministic tie-breaking is reproducible.

In [5]:
t0 = time.time()
louvain_rows = []
for domain, g in domain_graphs.items():
    if g.number_of_edges() == 0:
        louvain_rows.append({"domain": domain, "n_communities": 0, "modularity": float("nan"), "largest_community_frac": 0.0})
        continue
    partition = community_louvain.best_partition(g, weight="weight", random_state=SEED)
    mod = community_louvain.modularity(partition, g, weight="weight")
    sizes = Counter(partition.values())
    largest = max(sizes.values()) if sizes else 0
    louvain_rows.append(
        {
            "domain": domain,
            "n_communities": len(sizes),
            "modularity": float(mod),
            "largest_community_frac": largest / g.number_of_nodes(),
        }
    )
louvain_df = pd.DataFrame(louvain_rows).sort_values("modularity", ascending=False).reset_index(drop=True)
louvain_df.to_csv(TABLES / "louvain_per_domain.csv", index=False)
print(louvain_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
order = louvain_df.sort_values("modularity", ascending=False)["domain"]
sns.barplot(data=louvain_df, x="domain", y="modularity", order=order, color="#55a868", ax=ax)
ax.set(title="Louvain modularity per domain", xlabel="", ylabel="Modularity Q")
ax.tick_params(axis="x", rotation=45)
for label in ax.get_xticklabels():
    label.set_ha("right")
plt.tight_layout()
fig.savefig(FIGS / "louvain_per_domain.png", dpi=180, bbox_inches="tight")
plt.close(fig)
wall_clock["louvain"] = time.time() - t0

           domain  n_communities  modularity  largest_community_frac
       healthcare              7      0.4519                  0.2441
       literature              6      0.3111                  0.3210
           amazon              5      0.2708                  0.3009
 computer_science              6      0.2521                  0.3468
            legal              5      0.2296                  0.4643
          physics              4      0.2217                  0.2748
        chemistry              5      0.1809                  0.2739
materials_science              4      0.1706                  0.3473
          biology              5      0.1635                  0.2603
         medicine              4      0.1588                  0.2987


## 5. Centrality ↔ difficulty correlation

For each question we compute the mean PageRank and mean betweenness of its keywords (restricted to keywords present in the global graph — i.e. DF ≥ 3). Questions with zero retained keywords are excluded. We then compute Spearman r against an ordinal encoding of difficulty (easy=0, medium=1, hard=2).

**Directional hypothesis** (from CP2's reasoning-complexity finding): harder questions invoke rarer / more peripheral keywords, so we *expect* a weak negative correlation between centrality and difficulty. This would indicate that hard questions touch the long-tail of the vocabulary.

In [6]:
t0 = time.time()
difficulty_ord = df["level"].map({"easy": 0, "medium": 1, "hard": 2}).to_numpy()

pr_means, bc_means = [], []
node_set = set(global_g.nodes())
for toks in df["tokens"]:
    keep = [t for t in set(toks) if t in node_set]
    pr_means.append(float(np.mean([pr[t] for t in keep])) if keep else np.nan)
    bc_means.append(float(np.mean([bc[t] for t in keep])) if keep else np.nan)
df["pagerank_mean"] = pr_means
df["betweenness_mean"] = bc_means
valid = df.dropna(subset=["pagerank_mean", "betweenness_mean"])
print("valid questions for centrality correlation:", len(valid), "/", len(df))

rho_pr, p_pr = spearmanr(valid["pagerank_mean"], difficulty_ord[valid.index])
rho_bc, p_bc = spearmanr(valid["betweenness_mean"], difficulty_ord[valid.index])
print(f"Spearman r (PageRank vs difficulty)   : {rho_pr:+.4f}  p={p_pr:.2e}")
print(f"Spearman r (Betweenness vs difficulty): {rho_bc:+.4f}  p={p_bc:.2e}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, col, rho, pv, label in zip(
    axes,
    ["pagerank_mean", "betweenness_mean"],
    [rho_pr, rho_bc],
    [p_pr, p_bc],
    ["PageRank", "Betweenness"],
):
    sns.boxplot(data=valid, x="level", y=col, order=["easy", "medium", "hard"], hue="level",
                palette={"easy": "#2ca02c", "medium": "#ff7f0e", "hard": "#d62728"}, legend=False, ax=ax)
    ax.set(title=f"Mean {label} by difficulty  (ρ={rho:+.3f}, p={pv:.1e})", xlabel="", ylabel=label)
plt.tight_layout()
fig.savefig(FIGS / "pagerank_vs_difficulty.png", dpi=180, bbox_inches="tight")
plt.close(fig)
wall_clock["centrality_corr"] = time.time() - t0

valid questions for centrality correlation: 1740 / 1740
Spearman r (PageRank vs difficulty)   : -0.1600  p=1.89e-11
Spearman r (Betweenness vs difficulty): -0.1314  p=3.77e-08


## 6. Word-frequency baseline vs embeddings (RQ1 link)

A TF-IDF vectoriser on the raw tokenised questions (unigrams, min_df=3) feeds a Logistic Regression (`class_weight='balanced'`) over 5-fold stratified CV on difficulty. Macro-F1 is directly comparable to RQ1's `combined_lr` macro-F1 (0.839) and `embeddings_lr` macro-F1 (0.832).

In [7]:
t0 = time.time()
texts = df["question"].tolist()
y = df["level"].to_numpy()
tfidf = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, min_df=3, lowercase=False)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_scores = []
for tr, te in skf.split(texts, y):
    X_tr = tfidf.fit_transform([texts[i] for i in tr])
    X_te = tfidf.transform([texts[i] for i in te])
    clf = LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced")
    clf.fit(X_tr, y[tr])
    fold_scores.append(f1_score(y[te], clf.predict(X_te), average="macro"))
tfidf_macro_f1 = float(np.mean(fold_scores))
print(f"TF-IDF + LR macro-F1: {tfidf_macro_f1:.4f} (fold std {np.std(fold_scores):.4f})")
wall_clock["tfidf_baseline"] = time.time() - t0

TF-IDF + LR macro-F1: 0.9260 (fold std 0.0080)


## 7. Consolidated `metrics.json`

In [8]:
metrics = {
    "n_questions": int(len(df)),
    "n_domains": int(df["domain"].nunique()),
    "tokenization": {
        "type": "unigrams_alpha_len>=3_lowercase",
        "stopwords": {"source": "sklearn.ENGLISH_STOP_WORDS", "count": len(ENGLISH_STOP_WORDS), "extended_by": len(extra_stop)},
        "avg_tokens_per_question": float(df["tokens"].apply(len).mean()),
        "window": "question_level_document",
        "min_df": 3,
    },
    "per_domain_graph_stats": stats_df.to_dict(orient="records"),
    "global_graph": {
        "nodes": global_g.number_of_nodes(),
        "edges": global_g.number_of_edges(),
        "top15_pagerank": [{"keyword": k, "value": float(v)} for k, v in pr_top.items()],
        "top15_betweenness": [{"keyword": k, "value": float(v)} for k, v in bc_top.items()],
    },
    "louvain_per_domain": louvain_df.to_dict(orient="records"),
    "centrality_difficulty": {
        "pagerank":   {"spearman_r": float(rho_pr), "p_value": float(p_pr), "n": int(len(valid))},
        "betweenness": {"spearman_r": float(rho_bc), "p_value": float(p_bc), "n": int(len(valid))},
    },
    "tfidf_baseline_macro_f1": {
        "mean": tfidf_macro_f1,
        "std": float(np.std(fold_scores)),
        "folds": fold_scores,
        "comparison": {
            "rq1_embeddings_lr": 0.8315,
            "rq1_combined_lr": 0.8392,
            "delta_vs_embeddings_lr": tfidf_macro_f1 - 0.8315,
        },
    },
    "wall_clock_seconds": wall_clock,
}
(HANDOFF / "metrics.json").write_text(json.dumps(metrics, indent=2))
print(f"wrote {HANDOFF / 'metrics.json'}")
print(json.dumps({"rho_pr": rho_pr, "rho_bc": rho_bc, "tfidf_f1": tfidf_macro_f1,
                  "median_modularity": float(louvain_df["modularity"].median()),
                  "global_nodes": global_g.number_of_nodes(),
                  "global_edges": global_g.number_of_edges()}, indent=2))

wrote /home/ybai/CSCE676/handoff/rq2_graph_mining/metrics.json
{
  "rho_pr": -0.16003468543015467,
  "rho_bc": -0.13140341202569938,
  "tfidf_f1": 0.925989498512409,
  "median_modularity": 0.225629515422321,
  "global_nodes": 1675,
  "global_edges": 104674
}


## Wall-clock summary

In [9]:
print(pd.Series(wall_clock).round(2).to_string())

tokenize              0.02
per_domain_graphs     0.95
global_centrality    10.14
louvain               0.45
centrality_corr       0.24
tfidf_baseline        0.19
